
# Silver Sales Transformation

Transforms the Bronze sales dataset into a clean,
analytics-ready Silver Delta table.

Source:

`genai_copilot.bronze.sales_raw`

Target:

`genai_copilot.silver.sales`

Transformation steps:

1. Read Bronze
2. Inspect data quality
3. Cast data types
4. Remove duplicates
5. Handle missing values
6. Validate business rules
7. Calculate derived metrics
8. Add date dimensions
9. Write Silver Delta table
10. Validate the result

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    DoubleType,
    DateType,
)

In [0]:
BRONZE_TABLE = "genai_copilot.bronze.sales_raw"
SILVER_TABLE = "genai_copilot.silver.sales"

print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))

display(bronze_df.limit(10))

In [0]:
bronze_df.printSchema()

In [0]:
quality_summary = bronze_df.select(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_id"),

    F.sum(
        F.when(F.col("quantity") <= 0, 1).otherwise(0)
    ).alias("invalid_quantity"),

    F.sum(
        F.when(
            (F.col("discount") < 0) |
            (F.col("discount") > 1),
            1
        ).otherwise(0)
    ).alias("invalid_discount"),

    F.sum(
        F.when(F.col("revenue") < 0, 1).otherwise(0)
    ).alias("negative_revenue"),

    F.sum(
        F.when(F.col("cost") < 0, 1).otherwise(0)
    ).alias("negative_cost"),

    F.sum(
        F.when(F.col("customer_name").isNull(), 1).otherwise(0)
    ).alias("missing_customer_name")
)

display(quality_summary)

In [0]:
typed_df = (
    bronze_df
    .withColumn("order_id", F.col("order_id").cast(StringType()))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("customer_id", F.col("customer_id").cast(StringType()))
    .withColumn("customer_name", F.col("customer_name").cast(StringType()))
    .withColumn("region", F.col("region").cast(StringType()))
    .withColumn("country", F.col("country").cast(StringType()))
    .withColumn("product_id", F.col("product_id").cast(StringType()))
    .withColumn("product_name", F.col("product_name").cast(StringType()))
    .withColumn("category", F.col("category").cast(StringType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
    .withColumn("discount", F.col("discount").cast(DoubleType()))
    .withColumn("revenue", F.col("revenue").cast(DoubleType()))
    .withColumn("cost", F.col("cost").cast(DoubleType()))
    .withColumn("profit", F.col("profit").cast(DoubleType()))
    .withColumn("sales_channel", F.col("sales_channel").cast(StringType()))
    .withColumn("order_status", F.col("order_status").cast(StringType()))
)

In [0]:
standardized_df = (
    typed_df
    .withColumn(
        "region",
        F.initcap(F.trim(F.col("region")))
    )
    .withColumn(
        "country",
        F.initcap(F.trim(F.col("country")))
    )
    .withColumn(
        "category",
        F.initcap(F.trim(F.col("category")))
    )
    .withColumn(
        "sales_channel",
        F.initcap(F.trim(F.col("sales_channel")))
    )
    .withColumn(
        "order_status",
        F.initcap(F.trim(F.col("order_status")))
    )
    .withColumn(
        "customer_name",
        F.when(
            F.col("customer_name").isNull() |
            (F.trim(F.col("customer_name")) == ""),
            F.lit("Unknown Customer")
        ).otherwise(F.trim(F.col("customer_name")))
    )
)

In [0]:
deduplicated_df = standardized_df.dropDuplicates(["order_id"])

print("Rows after deduplication:", deduplicated_df.count())

In [0]:
valid_df = deduplicated_df.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("order_date").isNotNull()) &
    (F.col("quantity") > 0) &
    (F.col("unit_price") >= 0) &
    (F.col("discount") >= 0) &
    (F.col("discount") <= 1) &
    (F.col("revenue") >= 0) &
    (F.col("cost") >= 0)
)

print("Rows after business-rule validation:", valid_df.count())

In [0]:
revenue_check_df = valid_df.withColumn(
    "expected_revenue",
    F.col("quantity")
    * F.col("unit_price")
    * (1 - F.col("discount"))
)

In [0]:
revenue_check_df = revenue_check_df.withColumn(
    "revenue_difference",
    F.abs(
        F.col("revenue") - F.col("expected_revenue")
    )
)

display(
    revenue_check_df.select(
        "order_id",
        "revenue",
        "expected_revenue",
        "revenue_difference"
    ).orderBy(
        F.desc("revenue_difference")
    ).limit(20)
)

In [0]:
derived_df = (
    revenue_check_df
    .withColumn(
        "profit_margin",
        F.when(
            F.col("revenue") > 0,
            F.col("profit") / F.col("revenue")
        ).otherwise(F.lit(0.0))
    )
)

In [0]:
derived_df = (
    derived_df
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("quarter", F.quarter("order_date"))
)

In [0]:
silver_df = derived_df.select(
    "order_id",
    "order_date",
    "customer_id",
    "customer_name",
    "region",
    "country",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "unit_price",
    "discount",
    "revenue",
    "cost",
    "profit",
    "profit_margin",
    "sales_channel",
    "order_status",
    "year",
    "month",
    "quarter",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
silver_df.printSchema()

display(
    silver_df.limit(20)
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created Silver table: {SILVER_TABLE}")

In [0]:
display(
    spark.sql("""
        SELECT COUNT(*) AS silver_row_count
        FROM genai_copilot.silver.sales
    """)
)

In [0]:
%sql
SELECT COUNT(*) AS invalid_order_ids
FROM genai_copilot.silver.sales
WHERE order_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS invalid_quantities
FROM genai_copilot.silver.sales
WHERE quantity <= 0;

In [0]:
%sql
SELECT COUNT(*) AS invalid_discounts
FROM genai_copilot.silver.sales
WHERE discount < 0
   OR discount > 1;

In [0]:
%sql
SELECT COUNT(*) AS invalid_revenue
FROM genai_copilot.silver.sales
WHERE revenue < 0;

In [0]:
%sql
SELECT
    MIN(profit_margin) AS min_margin,
    MAX(profit_margin) AS max_margin
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    MIN(month) AS min_month,
    MAX(month) AS max_month,
    MIN(quarter) AS min_quarter,
    MAX(quarter) AS max_quarter
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS occurrences
FROM genai_copilot.silver.sales
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    COUNT(*) AS unknown_customers
FROM genai_copilot.silver.sales
WHERE customer_name = 'Unknown Customer';

In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit_margin), 4) AS avg_profit_margin,
    ROUND(AVG(revenue), 2) AS avg_order_revenue
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT COUNT(*)
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_orders
FROM (
    SELECT order_id
    FROM genai_copilot.silver.sales
    GROUP BY order_id
    HAVING COUNT(*) > 1
);